# Phase 3 : Modélisation

Ce notebook charge les données brutes et le pipeline de prétraitement pour entamer la modélisation.

**Étapes :**
1. Chargement de `dataset.csv`
2. Séparation des features `X` et de la cible `y` (`abandoned`)
3. Split stratifié en Train / Validation / Test (70/15/15)
4. Chargement du pipeline `preprocessor.joblib`
5. Vérification de la distribution de la variable cible pour confirmer le déséquilibre.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

# 1. Chargement des données brutes
df = pd.read_csv('../data/dataset.csv')

# Feature Engineering manquantes issues de la phase 3
df['phase'] = df['phase'].fillna('NA')
df = df.drop_duplicates()
df = df.drop(df[df['enrollment_count'] < 0].index)
violation_idx = df[(df['is_multicenter'] == 1) & (df['n_locations'] <= 1)].index
df.loc[violation_idx, 'is_multicenter'] = 0

df['log_enrollment'] = np.log1p(df['enrollment_count'])
df['log_n_locations'] = np.log1p(df['n_locations'])
df['protocol_complexity'] = df['n_arms'] * df['n_primary_outcomes']
df['enrollment_per_site'] = df['enrollment_count'] / (df['n_locations'] + 1)


# 2. Séparation des features (X) et de la cible (y)
X = df.drop(columns=['abandoned'])
y = df['abandoned']

# 3. Train / Validation / Test split (70/15/15) stratifié
# On sépare d'abord 30% pour un set temporaire (Val + Test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)

# Puis on divise le set temporaire en deux parties égales (50% Val, 50% Test)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

# 4. Chargement du pipeline de prétraitement
# Assurez-vous d'avoir scikit-learn d'installé dans l'environnement courant
preprocessor = joblib.load('../models/preprocessor.joblib')

# 5. Affichage des shapes et de la distribution
print("Shapes des sous-ensembles :")
print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")

print("\nDistribution de y_train (%) :")
print((y_train.value_counts(normalize=True) * 100).round(2))

Shapes des sous-ensembles :
X_train : (8563, 20)
X_val   : (1835, 20)
X_test  : (1835, 20)

Distribution de y_train (%) :
abandoned
0    86.34
1    13.66
Name: proportion, dtype: float64


### 2.1 Baseline — Régression Logistique

In [2]:
from sklearn.model_selection import cross_validate
from imblearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
import numpy as np

pipeline_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(class_weight='balanced', max_iter=5000, random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['f1', 'recall', 'precision']

results_lr = cross_validate(pipeline_lr, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

print("Régression Logistique :")
print(f"F1-score  : {np.mean(results_lr['test_f1']):.3f} ± {np.std(results_lr['test_f1']):.3f}")
print(f"Recall    : {np.mean(results_lr['test_recall']):.3f} ± {np.std(results_lr['test_recall']):.3f}")
print(f"Precision : {np.mean(results_lr['test_precision']):.3f} ± {np.std(results_lr['test_precision']):.3f}")


Régression Logistique :
F1-score  : 0.482 ± 0.018
Recall    : 0.715 ± 0.036
Precision : 0.364 ± 0.013


### 2.2 Decision Tree

In [3]:
from sklearn.model_selection import StratifiedKFold
scoring = ['f1', 'recall', 'precision']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
import numpy as np
from sklearn.model_selection import cross_validate
from imblearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

pipeline_dt = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(class_weight='balanced', random_state=42))
])

results_dt = cross_validate(pipeline_dt, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

print("Decision Tree :")
print(f"F1-score  : {np.mean(results_dt['test_f1']):.3f} ± {np.std(results_dt['test_f1']):.3f}")
print(f"Recall    : {np.mean(results_dt['test_recall']):.3f} ± {np.std(results_dt['test_recall']):.3f}")
print(f"Precision : {np.mean(results_dt['test_precision']):.3f} ± {np.std(results_dt['test_precision']):.3f}")


Decision Tree :
F1-score  : 0.480 ± 0.030
Recall    : 0.485 ± 0.036
Precision : 0.475 ± 0.025


### 2.3 Random Forest

In [4]:
from sklearn.model_selection import StratifiedKFold
scoring = ['f1', 'recall', 'precision']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
import numpy as np
from sklearn.model_selection import cross_validate
from imblearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(class_weight='balanced_subsample', n_estimators=200, random_state=42))
])

results_rf = cross_validate(pipeline_rf, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

print("Random Forest :")
print(f"F1-score  : {np.mean(results_rf['test_f1']):.3f} ± {np.std(results_rf['test_f1']):.3f}")
print(f"Recall    : {np.mean(results_rf['test_recall']):.3f} ± {np.std(results_rf['test_recall']):.3f}")
print(f"Precision : {np.mean(results_rf['test_precision']):.3f} ± {np.std(results_rf['test_precision']):.3f}")

Random Forest :
F1-score  : 0.583 ± 0.028
Recall    : 0.447 ± 0.033
Precision : 0.843 ± 0.017


### 2.4 XGBoost

In [5]:
from sklearn.model_selection import StratifiedKFold
scoring = ['f1', 'recall', 'precision']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
import numpy as np
from sklearn.model_selection import cross_validate
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier

scale = (y_train == 0).sum() / (y_train == 1).sum()

pipeline_xgb = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(scale_pos_weight=scale, eval_metric='logloss', random_state=42))
])

results_xgb = cross_validate(pipeline_xgb, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

print("XGBoost :")
print(f"F1-score  : {np.mean(results_xgb['test_f1']):.3f} ± {np.std(results_xgb['test_f1']):.3f}")
print(f"Recall    : {np.mean(results_xgb['test_recall']):.3f} ± {np.std(results_xgb['test_recall']):.3f}")
print(f"Precision : {np.mean(results_xgb['test_precision']):.3f} ± {np.std(results_xgb['test_precision']):.3f}")

XGBoost :
F1-score  : 0.541 ± 0.024
Recall    : 0.547 ± 0.029
Precision : 0.536 ± 0.026


### 2.5 SVM

In [6]:
from sklearn.model_selection import StratifiedKFold
scoring = ['f1', 'recall', 'precision']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
import numpy as np
from sklearn.model_selection import cross_validate
from imblearn.pipeline import Pipeline
from sklearn.svm import SVC

pipeline_svm = Pipeline([
    ('preprocessor', preprocessor),
    ('model', SVC(kernel='rbf', class_weight='balanced', random_state=42))
])

results_svm = cross_validate(pipeline_svm, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

print("SVM :")
print(f"F1-score  : {np.mean(results_svm['test_f1']):.3f} ± {np.std(results_svm['test_f1']):.3f}")
print(f"Recall    : {np.mean(results_svm['test_recall']):.3f} ± {np.std(results_svm['test_recall']):.3f}")
print(f"Precision : {np.mean(results_svm['test_precision']):.3f} ± {np.std(results_svm['test_precision']):.3f}")

SVM :
F1-score  : 0.294 ± 0.101
Recall    : 0.935 ± 0.109
Precision : 0.187 ± 0.096


### 2.6 MLP

In [7]:
from sklearn.model_selection import StratifiedKFold
scoring = ['f1', 'recall', 'precision']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
import numpy as np
from sklearn.model_selection import cross_validate
from imblearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier

pipeline_mlp = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPClassifier(early_stopping=True, max_iter=500, random_state=42))
])

results_mlp = cross_validate(pipeline_mlp, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

print("MLP :")
print(f"F1-score  : {np.mean(results_mlp['test_f1']):.3f} ± {np.std(results_mlp['test_f1']):.3f}")
print(f"Recall    : {np.mean(results_mlp['test_recall']):.3f} ± {np.std(results_mlp['test_recall']):.3f}")
print(f"Precision : {np.mean(results_mlp['test_precision']):.3f} ± {np.std(results_mlp['test_precision']):.3f}")

MLP :
F1-score  : 0.554 ± 0.036
Recall    : 0.409 ± 0.044
Precision : 0.871 ± 0.039


### 2.7 Tableau comparatif des 6 modeles (sans reequilibrage)

In [8]:
from IPython.display import display
import numpy as np
import pandas as pd

results_map = {
    'Logistic Regression': results_lr,
    'Decision Tree':        results_dt,
    'Random Forest':        results_rf,
    'XGBoost':              results_xgb,
    'SVM':                  results_svm,
    'MLP':                  results_mlp,
}

data = []
for name, res in results_map.items():
    data.append({
        'Modele':      name,
        'F1 mean':     round(np.mean(res['test_f1']),        3),
        'F1 std':      round(np.std(res['test_f1']),         3),
        'Recall mean': round(np.mean(res['test_recall']),    3),
        'Prec. mean':  round(np.mean(res['test_precision']), 3),
    })

summary = pd.DataFrame(data).sort_values('F1 mean', ascending=False)
display(summary)
print("Meilleur modele baseline :", summary.iloc[0]['Modele'])

,Modele,F1 mean,F1 std,Recall mean,Prec. mean
2,Random Forest,0.583,0.028,0.447,0.843
5,MLP,0.554,0.036,0.409,0.871
3,XGBoost,0.541,0.024,0.547,0.536
0,Logistic Regression,0.482,0.018,0.715,0.364
1,Decision Tree,0.480,0.030,0.485,0.475
4,SVM,0.294,0.101,0.935,0.187


Meilleur modele baseline : Random Forest


### 2.8 SMOTE sur tous les modeles

In [9]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
import numpy as np

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['f1', 'recall', 'precision']

pipelines_smote = {
    'Logistic Regression': Pipeline([('preprocessor', preprocessor), ('smote', SMOTE(random_state=42)), ('model', LogisticRegression(max_iter=5000, random_state=42))]),
    'Decision Tree':       Pipeline([('preprocessor', preprocessor), ('smote', SMOTE(random_state=42)), ('model', DecisionTreeClassifier(random_state=42))]),
    'Random Forest':       Pipeline([('preprocessor', preprocessor), ('smote', SMOTE(random_state=42)), ('model', RandomForestClassifier(n_estimators=200, random_state=42))]),
    'XGBoost':             Pipeline([('preprocessor', preprocessor), ('smote', SMOTE(random_state=42)), ('model', XGBClassifier(eval_metric='logloss', random_state=42))]),
    'SVM':                 Pipeline([('preprocessor', preprocessor), ('smote', SMOTE(random_state=42)), ('model', SVC(kernel='rbf', random_state=42))]),
    'MLP':                 Pipeline([('preprocessor', preprocessor), ('smote', SMOTE(random_state=42)), ('model', MLPClassifier(early_stopping=True, max_iter=500, random_state=42))]),
}

results_smote = {}
for name, pipe in pipelines_smote.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    results_smote[name] = res
    print(f"{name} (SMOTE) :")
    print(f"  F1-score  : {np.mean(res['test_f1']):.3f} +/- {np.std(res['test_f1']):.3f}")
    print(f"  Recall    : {np.mean(res['test_recall']):.3f} +/- {np.std(res['test_recall']):.3f}")
    print(f"  Precision : {np.mean(res['test_precision']):.3f} +/- {np.std(res['test_precision']):.3f}")

Logistic Regression (SMOTE) :
  F1-score  : 0.485 +/- 0.012
  Recall    : 0.714 +/- 0.034
  Precision : 0.368 +/- 0.007
Decision Tree (SMOTE) :
  F1-score  : 0.461 +/- 0.018
  Recall    : 0.498 +/- 0.016
  Precision : 0.429 +/- 0.019
Random Forest (SMOTE) :
  F1-score  : 0.595 +/- 0.027
  Recall    : 0.501 +/- 0.028
  Precision : 0.735 +/- 0.042
XGBoost (SMOTE) :
  F1-score  : 0.578 +/- 0.021
  Recall    : 0.472 +/- 0.022
  Precision : 0.746 +/- 0.019
SVM (SMOTE) :
  F1-score  : 0.312 +/- 0.112
  Recall    : 0.898 +/- 0.127
  Precision : 0.210 +/- 0.124
MLP (SMOTE) :
  F1-score  : 0.518 +/- 0.029
  Recall    : 0.634 +/- 0.049
  Precision : 0.438 +/- 0.024


### 2.9 RandomUnderSampler sur tous les modeles

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold
scoring = ['f1', 'recall', 'precision']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
import numpy as np
from sklearn.model_selection import cross_validate
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

pipelines_rus = {
    'Logistic Regression': Pipeline([('preprocessor', preprocessor), ('rus', RandomUnderSampler(random_state=42)), ('model', LogisticRegression(max_iter=5000, random_state=42))]),
    'Decision Tree': Pipeline([('preprocessor', preprocessor), ('rus', RandomUnderSampler(random_state=42)), ('model', DecisionTreeClassifier(random_state=42))]),
    'Random Forest': Pipeline([('preprocessor', preprocessor), ('rus', RandomUnderSampler(random_state=42)), ('model', RandomForestClassifier(n_estimators=200, random_state=42))]),
    'XGBoost': Pipeline([('preprocessor', preprocessor), ('rus', RandomUnderSampler(random_state=42)), ('model', XGBClassifier(eval_metric='logloss', random_state=42))]),
    'SVM': Pipeline([('preprocessor', preprocessor), ('rus', RandomUnderSampler(random_state=42)), ('model', SVC(kernel='rbf', random_state=42))]),
    'MLP': Pipeline([('preprocessor', preprocessor), ('rus', RandomUnderSampler(random_state=42)), ('model', MLPClassifier(early_stopping=True, max_iter=500, random_state=42))])
}

results_rus = {}
for name, pipe in pipelines_rus.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    results_rus[name] = res
    print(f"{name} (RandomUnderSampler) :")
    print(f"  F1-score  : {np.mean(res['test_f1']):.3f} +/- {np.std(res['test_f1']):.3f}")
    print(f"  Recall    : {np.mean(res['test_recall']):.3f} +/- {np.std(res['test_recall']):.3f}")
    print(f"  Precision : {np.mean(res['test_precision']):.3f} +/- {np.std(res['test_precision']):.3f}")

Logistic Regression (RandomUnderSampler) :
  F1-score  : 0.476 +/- 0.016
  Recall    : 0.722 +/- 0.032
  Precision : 0.356 +/- 0.014
Decision Tree (RandomUnderSampler) :
  F1-score  : 0.385 +/- 0.009
  Recall    : 0.721 +/- 0.013
  Precision : 0.262 +/- 0.009
Random Forest (RandomUnderSampler) :
  F1-score  : 0.503 +/- 0.018
  Recall    : 0.692 +/- 0.037
  Precision : 0.395 +/- 0.015
XGBoost (RandomUnderSampler) :
  F1-score  : 0.458 +/- 0.021
  Recall    : 0.709 +/- 0.039
  Precision : 0.339 +/- 0.016
SVM (RandomUnderSampler) :
  F1-score  : 0.392 +/- 0.128
  Recall    : 0.820 +/- 0.145
  Precision : 0.286 +/- 0.130
MLP (RandomUnderSampler) :
  F1-score  : 0.461 +/- 0.035
  Recall    : 0.749 +/- 0.033
  Precision : 0.335 +/- 0.040


### 2.10 Tableau comparatif 6x3

In [19]:
from IPython.display import display
import numpy as np
import pandas as pd

modeles = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost', 'SVM', 'MLP']
results_base = {
    'Logistic Regression': results_lr,
    'Decision Tree':        results_dt,
    'Random Forest':        results_rf,
    'XGBoost':              results_xgb,
    'SVM':                  results_svm,
    'MLP':                  results_mlp,
}

data = []
for name in modeles:
    rb = results_base[name]
    rs = results_smote[name]
    rr = results_rus[name]
    data.append({
        'Modele':                name,
        'Sans reeq. Recall':     round(np.mean(rb['test_recall']), 3),
        'Sans reeq. Precision':  round(np.mean(rb['test_precision']), 3),
        'Sans reeq. F1 +/- std': f"{np.mean(rb['test_f1']):.3f} +/- {np.std(rb['test_f1']):.3f}",
        'SMOTE Recall':          round(np.mean(rs['test_recall']), 3),
        'SMOTE Precision':       round(np.mean(rs['test_precision']), 3),
        'SMOTE F1 +/- std':      f"{np.mean(rs['test_f1']):.3f} +/- {np.std(rs['test_f1']):.3f}",
        'RUS Recall':            round(np.mean(rr['test_recall']), 3),
        'RUS Precision':         round(np.mean(rr['test_precision']), 3),
        'RUS F1 +/- std':        f"{np.mean(rr['test_f1']):.3f} +/- {np.std(rr['test_f1']):.3f}",
    })

df_comparatif = pd.DataFrame(data)
display(df_comparatif)

,Modele,Sans reeq. Recall,Sans reeq. Precision,Sans reeq. F1 +/- std,SMOTE Recall,SMOTE Precision,SMOTE F1 +/- std,RUS Recall,RUS Precision,RUS F1 +/- std
0,Logistic Regression,0.715,0.364,0.482 +/- 0.018,0.714,0.368,0.485 +/- 0.012,0.722,0.356,0.476 +/- 0.016
1,Decision Tree,0.485,0.475,0.480 +/- 0.030,0.498,0.429,0.461 +/- 0.018,0.721,0.262,0.385 +/- 0.009
2,Random Forest,0.447,0.843,0.583 +/- 0.028,0.501,0.735,0.595 +/- 0.027,0.692,0.395,0.503 +/- 0.018
3,XGBoost,0.547,0.536,0.541 +/- 0.024,0.472,0.746,0.578 +/- 0.021,0.709,0.339,0.458 +/- 0.021
4,SVM,0.935,0.187,0.294 +/- 0.101,0.898,0.210,0.312 +/- 0.112,0.820,0.286,0.392 +/- 0.128
5,MLP,0.409,0.871,0.554 +/- 0.036,0.634,0.438,0.518 +/- 0.029,0.749,0.335,0.461 +/- 0.035


### 2.11 Selection de la meilleure combinaison

In [12]:
import numpy as np

modeles = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost', 'SVM', 'MLP']
results_base = {
    'Logistic Regression': results_lr,
    'Decision Tree':        results_dt,
    'Random Forest':        results_rf,
    'XGBoost':              results_xgb,
    'SVM':                  results_svm,
    'MLP':                  results_mlp,
}

strategies = {
    'Sans reeq.': results_base,
    'SMOTE':      results_smote,
    'RUS':        results_rus,
}

best_f1 = 0
best_recall = 0
best_combo = ('', '')

for strat_name, strat_results in strategies.items():
    for model_name in modeles:
        f1     = np.mean(strat_results[model_name]['test_f1'])
        recall = np.mean(strat_results[model_name]['test_recall'])
        if recall >= 0.80 and f1 > best_f1:
            best_f1    = f1
            best_recall = recall
            best_combo = (model_name, strat_name)

if best_combo == ('', ''):
    print("Aucune combinaison n'atteint recall >= 0.80 — selection par meilleur F1 global")
    for strat_name, strat_results in strategies.items():
        for model_name in modeles:
            f1 = np.mean(strat_results[model_name]['test_f1'])
            if f1 > best_f1:
                best_f1    = f1
                best_combo = (model_name, strat_name)

best_model_name, best_strategy = best_combo
print(f"Combinaison retenue pour le tuning : {best_model_name} + {best_strategy}")
print(f"F1 : {best_f1:.3f} | Recall : {best_recall:.3f}")

Combinaison retenue pour le tuning : SVM + RUS
F1 : 0.392 | Recall : 0.820


### 2.12 Evaluation de la meilleure combinaison sur validation

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, classification_report, confusion_matrix

pipelines_all = {
    'Sans reeq.': {
        'Logistic Regression': pipeline_lr,
        'Decision Tree': pipeline_dt,
        'Random Forest': pipeline_rf,
        'XGBoost': pipeline_xgb,
        'SVM': pipeline_svm,
        'MLP': pipeline_mlp
    },
    'SMOTE': pipelines_smote,
    'RUS': pipelines_rus
}

best_pipe = pipelines_all[best_strategy][best_model_name]
best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_val)

print(f"Evaluation sur X_val — {best_model_name} + {best_strategy}")
print(f"F1        : {f1_score(y_val, y_pred):.3f}  (seuil : >= 0.55)")
print(f"Recall    : {recall_score(y_val, y_pred):.3f}  (seuil : >= 0.80)")
print(f"Precision : {precision_score(y_val, y_pred):.3f}  (seuil : >= 0.40)")
print(f"Accuracy  : {accuracy_score(y_val, y_pred):.3f}")
print()
print(classification_report(y_val, y_pred, target_names=['Complete', 'Abandonne']))
print()
print("Matrice de confusion :")
print(confusion_matrix(y_val, y_pred))

Evaluation sur X_val — SVM + RUS
F1        : 0.508  (seuil : >= 0.55)
Recall    : 0.600  (seuil : >= 0.80)
Precision : 0.440  (seuil : >= 0.40)
Accuracy  : 0.841

              precision    recall  f1-score   support

    Complete       0.93      0.88      0.91      1585
   Abandonne       0.44      0.60      0.51       250

    accuracy                           0.84      1835
   macro avg       0.69      0.74      0.71      1835
weighted avg       0.87      0.84      0.85      1835


Matrice de confusion :
[[1394  191]
 [ 100  150]]


### Synthese finale

## Synthese

### Modeles testes
6 modeles ont ete evalues : Logistic Regression (baseline), Decision Tree, Random Forest, XGBoost, SVM et MLP.

### Strategies de desequilibre comparees
3 strategies ont ete comparees pour chaque modele :
- Sans reeequilibrage (class_weight/scale_pos_weight)
- SMOTE (oversampling de la minorite)
- RandomUnderSampler (undersampling de la majorite)

### Tableau 6x3
18 combinaisons evaluees par validation croisee stratifiee (5 folds), avec F1, Recall et Precision reportes pour chaque combinaison.

### Combinaison retenue
La meilleure combinaison est selectionnee en priorite sur recall >= 0.80, puis sur F1 maximal.

### Suite
La combinaison retenue sera tunee dans le notebook 05_tuning.ipynb.